# AI Powered E-Commerce Customer Intelligence System

## Problem Statement
We are junior data scientists. We will inspect a SQLite e-commerce database,
clean it, answer business questions with SQL, do EDA, build a churn model,
a small neural network, and a review sentiment model (Session 26 style).

In [2]:
### Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3 as sql
import warnings
warnings.filterwarnings('ignore')

In [3]:
conn = sql.connect('ecommerce_hackathon.db')
conn

In [4]:
q = '''
SELECT name FROM sqlite_master
WHERE type = 'table'
'''
pd.read_sql_query(q, conn)

,name
0,customers
1,products
2,orders
3,reviews


# Task A: Database Inspection and Cleaning 

In [6]:
def sql_query(q):
    return pd.read_sql_query(q, conn)
    
customers = sql_query("SELECT * FROM customers")
products  = sql_query("SELECT * FROM products")
orders    = sql_query("SELECT * FROM orders")
reviews   = sql_query("SELECT * FROM reviews")

In [7]:
print("customers", customers.shape)
print(customers.columns.tolist())
customers.head()

customers (8000, 9)
['customer_id', 'customer_name', 'age', 'gender', 'city', 'signup_date', 'membership_type', 'email', 'phone']


,customer_id,customer_name,age,gender,city,signup_date,membership_type,email,phone
0,1,Hassan Khan,20.0,Male,Karachi,2026-02-27,Standard,hassan.khan1@example.com,+92-3241-4744854
1,2,Maham Raza,33.0,Female,Rawalpindi,2025-10-17,Silver,maham.raza2@example.com,+92-3421-8078673
2,3,Ahmed Khan,25.0,Male,Karachi,2022-07-11,Standard,ahmed.khan3@example.com,+92-3139-9478454
3,4,Salman Khan,26.0,Male,Lahore,2025-02-23,Standard,salman.khan4@example.com,+92-3193-8038374
4,5,Noor Awan,27.0,Female,Lahore,2025-04-21,Standard,noor.awan5@example.com,+92-3210-3678638


In [8]:
print("products", products.shape)
print(products.columns.tolist())
products.head()

products (1000, 6)
['product_id', 'product_name', 'category', 'brand', 'unit_price', 'stock_quantity']


,product_id,product_name,category,brand,unit_price,stock_quantity
0,1,Readings Exam Guide Premium,Books & Stationery,Readings,566.45,103
1,2,Samsung Webcam Classic,Electronics,Samsung,10567.23,138
2,3,CA Sports Water Bottle Classic,Sports & Fitness,CA Sports,6665.04,63
3,4,LU Spice Mix Classic,Groceries,LU,516.49,113
4,5,L'Oreal Shampoo 2026,Beauty,L'Oreal,2581.35,107


In [9]:
print("orders", orders.shape)
print(orders.columns.tolist())
orders.head()

orders (65000, 10)
['order_id', 'customer_id', 'product_id', 'order_date', 'quantity', 'unit_price', 'discount', 'payment_method', 'delivery_days', 'returned']


,order_id,customer_id,product_id,order_date,quantity,unit_price,discount,payment_method,delivery_days,returned
0,1,2367,389,2026-04-30,1,398.27,0.108,Cash on Delivery,1.0,0
1,2,2038,927,2026-02-09,2,44488.20,0.084,Debit/Credit Card,1.0,0
2,3,6285,909,2026-03-30,1,4719.91,0.125,Cash on Delivery,1.0,0
3,4,3714,117,2026-03-24,1,2053.64,0.118,Easypaisa,4.0,0
4,5,3021,733,2024-12-13,1,31378.77,0.087,Debit/Credit Card,2.0,0


In [10]:
print("reviews", reviews.shape)
print(reviews.columns.tolist())
reviews.head()

reviews (26000, 6)
['review_id', 'customer_id', 'product_id', 'review_date', 'rating', 'review_text']


,review_id,customer_id,product_id,review_date,rating,review_text
0,1,7210,787,2026-07-28,4,"I would buy this again, item feels durable. Re..."
1,2,5566,765,2026-05-16,5,"Very satisfied, seller description was accurat..."
2,3,6106,743,2025-10-21,4,"Very satisfied, quality feels premium. Good va..."
3,4,2735,90,2026-03-25,4,"Product arrived in great condition, quality fe..."
4,5,2124,197,2026-02-23,4,"Good experience overall, packing was neat. Wil..."


In [11]:
customers.info()
products.info()
orders.info()
reviews.info()

<class 'pandas.DataFrame'>
RangeIndex: 8000 entries, 0 to 7999
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   customer_id      8000 non-null   int64  
 1   customer_name    8000 non-null   str    
 2   age              7900 non-null   float64
 3   gender           8000 non-null   str    
 4   city             8000 non-null   str    
 5   signup_date      8000 non-null   str    
 6   membership_type  8000 non-null   str    
 7   email            8000 non-null   str    
 8   phone            8000 non-null   str    
dtypes: float64(1), int64(1), str(7)
memory usage: 1.2 MB
<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   product_id      1000 non-null   int64  
 1   product_name    1000 non-null   str    
 2   category        1000 non-null   str    
 3   brand           9

In [12]:
### Duplicates, categories, invalid numbers, dates
print("duplicate rows customers", customers.duplicated().sum())
print("duplicate rows products", products.duplicated().sum())
print("duplicate rows orders", orders.duplicated().sum())
print("duplicate rows reviews", reviews.duplicated().sum())

print(customers["gender"].value_counts())
print(customers["membership_type"].value_counts())
print(customers["city"].value_counts().head(20))

print(products["category"].value_counts())

print(orders["payment_method"].value_counts(dropna=False))
print(orders["returned"].value_counts())

print(orders["discount"].describe())
print("qty 0", (orders["quantity"] == 0).sum())
print("price < 0", (orders["unit_price"] < 0).sum())

print(reviews["rating"].value_counts().sort_index())


duplicate rows customers 0
duplicate rows products 0
duplicate rows orders 0
duplicate rows reviews 0
gender
Male      4517
Female    3483
Name: count, dtype: int64
membership_type
Standard    4629
Silver      1776
Gold        1113
Premium      482
Name: count, dtype: int64
city
Karachi       2149
Lahore        1461
Islamabad      798
Rawalpindi     658
Faisalabad     564
Multan         480
Peshawar       399
Hyderabad      318
Gujranwala     316
Quetta         237
Sialkot        234
Bahawalpur     159
Sargodha       157
KARACHI         10
karachi          7
 Lahore          6
lahore           5
LAHORE           4
 Karachi         3
MULTAN           3
Name: count, dtype: int64
category
Fashion               218
Electronics           179
Home & Kitchen        132
Beauty                130
Groceries             100
Sports & Fitness       87
Books & Stationery     74
Baby & Kids            62
Fashion                 3
electronics             3
ELECTRONICS             2
BEAUTY             

# cleaning

In [14]:
customers_clean = customers.copy()
products_clean  = products.copy()
orders_clean    = orders.copy()
reviews_clean   = reviews.copy()

# city names should be one spelling so city revenue is not split
customers_clean["city"] = customers_clean["city"].str.strip().str.title()

# age needed for churn; fill like Session 13 (do not drop 100 people)
customers_clean["age"] = customers_clean.groupby("gender")["age"].transform(
    lambda s: s.fillna(s.median())
)
customers_clean["signup_date"] = pd.to_datetime(customers_clean["signup_date"], errors="coerce")

products_clean["category"] = products_clean["category"].str.strip().str.title()
products_clean["brand"] = products_clean["brand"].fillna(
    products_clean["product_name"].str.split().str[0]
)

# negative price is a sign error; abs keeps revenue correct
orders_clean["unit_price"] = orders_clean["unit_price"].abs()

# qty 0 cannot be a real sale
orders_clean.loc[orders_clean["quantity"] == 0, "quantity"] = 1

orders_clean["payment_method"] = orders_clean["payment_method"].fillna(
    orders_clean["payment_method"].mode()[0]
)
orders_clean["delivery_days"] = orders_clean["delivery_days"].fillna(
    orders_clean["delivery_days"].median()
)

orders_clean["order_date"] = pd.to_datetime(orders_clean["order_date"], errors="coerce")
orders_clean["order_date"] = orders_clean["order_date"].fillna(orders_clean["order_date"].median())

reviews_clean["rating"] = reviews_clean["rating"].clip(1, 5)
reviews_clean["review_text"] = reviews_clean["review_text"].fillna("").str.strip()
reviews_clean["review_date"] = pd.to_datetime(reviews_clean["review_date"], errors="coerce")
reviews_clean["review_date"] = reviews_clean["review_date"].fillna(reviews_clean["review_date"].median())

print(customers_clean.isnull().sum())
print(products_clean.isnull().sum())
print(orders_clean.isnull().sum())

customer_id        0
customer_name      0
age                0
gender             0
city               0
signup_date        0
membership_type    0
email              0
phone              0
dtype: int64
product_id        0
product_name      0
category          0
brand             0
unit_price        0
stock_quantity    0
dtype: int64
order_id          0
customer_id       0
product_id        0
order_date        0
quantity          0
unit_price        0
discount          0
payment_method    0
delivery_days     0
returned          0
dtype: int64


# Task B: SQL Business Analysis

In [15]:
clean_conn = sql.connect("ecommerce_clean.db")

customers_clean.to_sql("customers", clean_conn, if_exists="replace", index=False)
products_clean.to_sql("products",  clean_conn, if_exists="replace", index=False)
orders_clean.to_sql("orders",      clean_conn, if_exists="replace", index=False)
reviews_clean.to_sql("reviews",    clean_conn, if_exists="replace", index=False)

26000

In [16]:
def sql_query(q):
    return pd.read_sql_query(q, clean_conn)